# Habits by the Numbers: Learning Curves and the Power Law of Practice

**CS474: Human Computer Interaction — Leveraging Habits for Intuition**

Interfaces feel "intuitive" when practiced actions become automatic.  Remarkably, this speedup follows a predictable mathematical pattern: the **power law of practice** (Newell & Rosenbloom, 1981) — the time to perform a task drops as a power function of the number of times you've done it:

$$ T_n = T_1 \cdot n^{-\alpha} $$

In this notebook you will:

1. Simulate a user practicing a UI task over many trials
2. Fit the power law to the data and interpret the learning-rate parameter
3. See why the curve looks like a straight line on log-log axes
4. Model what happens when a redesign *breaks* a learned habit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)

## Part 1: Simulate Practice

A user performs the same task (say, filing an expense report in a new app) 100 times.  Their time follows the power law, plus human variability.

In [ ]:
N_TRIALS = 100
T1, ALPHA = 40.0, 0.35        # first-trial time (s), learning rate
trials = np.arange(1, N_TRIALS + 1)

true_time = T1 * trials ** (-ALPHA)
observed = true_time * rng.lognormal(0, 0.12, N_TRIALS)  # multiplicative noise

plt.figure(figsize=(7, 3.5))
plt.plot(trials, observed, 'o', ms=4, alpha=0.5, label='observed trials')
plt.plot(trials, true_time, '-', color='tab:red', label='underlying power law')
plt.xlabel('trial number'); plt.ylabel('task time (s)')
plt.title('Learning curve: practice makes fast')
plt.legend()
plt.show()

## Part 2: Fit the Power Law

Taking logs of both sides turns the power law into a line:

$$ \log T_n = \log T_1 - \alpha \log n $$

so we can fit it with ordinary least squares (`np.polyfit`) on the logged data.

In [ ]:
slope, intercept = np.polyfit(np.log(trials), np.log(observed), 1)
alpha_hat, T1_hat = -slope, np.exp(intercept)
print(f"Estimated T1 = {T1_hat:.1f} s (true {T1}),  alpha = {alpha_hat:.3f} (true {ALPHA})")

plt.figure(figsize=(7, 3.5))
plt.loglog(trials, observed, 'o', ms=4, alpha=0.5, label='observed')
plt.loglog(trials, T1_hat * trials ** (-alpha_hat), '-', color='tab:red', label='fitted power law')
plt.xlabel('trial number (log scale)'); plt.ylabel('task time (s, log scale)')
plt.title('On log-log axes, the power law is a straight line')
plt.legend()
plt.show()

The parameter alpha is the **learning rate**: a steeper alpha means the interface converts practice into speed more efficiently.  Designs with consistent, repeated action patterns (same button, same place, same gesture) tend to have higher effective alpha — that consistency is what lets a habit form.

## Part 3: A Redesign Breaks the Habit

At trial 60, the app ships a redesign that moves the key buttons.  The user's habit no longer matches the interface: time jumps, and — worse — for a while they make *habit-capture errors* (executing the old, now-wrong action).  We model the disruption as a partial reset of learning.

In [ ]:
before = T1 * trials[:60] ** (-ALPHA)
# after redesign: learning restarts from trial ~1 but with some transfer (30 s, not 40 s)
after = 30.0 * np.arange(1, N_TRIALS - 60 + 1) ** (-ALPHA)
disrupted = np.concatenate([before, after]) * rng.lognormal(0, 0.12, N_TRIALS)

plt.figure(figsize=(7, 3.5))
plt.plot(trials, disrupted, 'o', ms=4, alpha=0.5)
plt.axvline(60, color='red', ls='--', label='redesign ships')
plt.xlabel('trial number'); plt.ylabel('task time (s)')
plt.title('A redesign that breaks a learned habit')
plt.legend()
plt.show()

## Your Turn

1. **How much practice is "intuitive"?**  Using the fitted curve, after how many trials does task time fall below 15 s?  Below 12 s?  Notice the diminishing returns — what does that imply for onboarding tutorials?
2. **Compare two designs.**  Simulate design X (T1 = 25, alpha = 0.15: easy to start, slow to master) versus design Y (T1 = 45, alpha = 0.45: hard to start, fast to master).  Plot both curves — after how many trials does Y win?  Which design should a walk-up kiosk use?  Which should an air-traffic-control console use?
3. **Design connection.**  This is Gourville's "curse of innovation" (from our readings) in miniature: the redesign in Part 3 might be objectively better at steady state, yet users experience a loss.  Using your Part 3 plot, estimate the *transition cost* (extra seconds summed over trials 60-100).  What could a designer do to reduce it (progressive rollout, spatial consistency, optional classic mode)?

## Reflection

Nintendo's World 1-1 (from the activity model) teaches by letting the player *practice* each mechanic in a safe spot before combining them — deliberately shaping this curve.  When you design your final project, ask: what will your users do 100 times, and does your design let that action become a habit?